# OpenSMILE ComParE13 feature extraction (ALI\_new/All)

This notebook is the same extraction pipeline you provided, with only these changes:

- `BALANCED_DATA_PATH` → `/mnt/d/PhD/ALS_PROJECT1/ALS_Diagnosis_Meta/Dataset/ALI_new/All`
- `DATASET_OUTPUT` → `/mnt/d/PhD/ALS_PROJECT1/ALS_Diagnosis_Meta/Results/OpenSMILE_ALI_new_All`


In [ ]:
import os
import numpy as np
import pandas as pd
from tqdm import tqdm
import pickle
import warnings
warnings.filterwarnings('ignore')

FEATURE_DIM = 6374
OPENSMILE_TIMEOUT_S = 60  # increase if some files are slow
RESUME_EXISTING = False  # set True only if you want to continue a partially-finished run

# ============================================
# PATHS
# ============================================
# First, let's define the extraction function here instead of importing
def extract_opensmile_features(wav_file):
    """
    Extract OpenSMILE features handling quoted 'unknown'
    Returns: numpy array of 6374 features or None if failed
    """
    import subprocess
    import tempfile
    
    OPENSMILE_PATH = "/mnt/d/PhD/ALS_Diagnosis_Meta/opensmile-3.0.2-linux-x86_64/bin/SMILExtract"
    CONFIG_PATH = "/mnt/d/PhD/ALS_Diagnosis_Meta/opensmile-3.0.2-linux-x86_64/config/is09-13/IS13_ComParE.conf"
    
    # Create temp output
    temp_csv = tempfile.NamedTemporaryFile(suffix='.csv', delete=False)
    temp_csv.close()
    output_csv = temp_csv.name
    
    try:
        # Run OpenSMILE
        cmd = [
            OPENSMILE_PATH,
            "-C", CONFIG_PATH,
            "-I", wav_file,
            "-csvoutput", output_csv,
            "-noconsoleoutput", "1",
            "-loglevel", "0"
        ]
        
        subprocess.run(cmd, capture_output=True, text=True, timeout=OPENSMILE_TIMEOUT_S, check=True)
        
        # Read the CSV content
        with open(output_csv, 'r') as f:
            content = f.read().strip()
        
        if not content:
            return None
        
        # Split by semicolon
        parts = content.split(';')
        
        # Remove quotes from each part and strip whitespace
        parts_clean = [p.strip().strip("'").strip('"') for p in parts]
        
        # First element is 'unknown' (label), skip it
        # Convert rest to float
        features = []
        for val in parts_clean[1:]:  # Skip the first element
            try:
                features.append(float(val))
            except ValueError:
                # If conversion fails, use 0
                features.append(0.0)
        
        return np.array(features, dtype=np.float32)
        
    except Exception as e:
        print(f"Error processing {os.path.basename(wav_file)}: {e}")
        return None
    finally:
        # Cleanup
        if os.path.exists(output_csv):
            os.remove(output_csv)

# INPUT (balanced raw wavs)
BALANCED_DATA_PATH = "/mnt/d/PhD/ALS_PROJECT1/ALS_Diagnosis_Meta/Dataset/ALI_new/All"
DATASET_OUTPUT = "/mnt/d/PhD/ALS_PROJECT1/ALS_Diagnosis_Meta/Results/OpenSMILE_ALI_new_All"
os.makedirs(DATASET_OUTPUT, exist_ok=True)

os.makedirs(DATASET_OUTPUT, exist_ok=True)

# ============================================
# DATASET CREATION - COMPLETE FUNCTION
# ============================================

def extract_all_subjects():
    """
    STEP 2: Extract features for ALL subjects and create complete dataset
    """
    print("="*80)
    print("STEP 2: EXTRACTING FEATURES FOR ALL SUBJECTS")
    print("="*80)
    
    # Initialize data structures
    subject_data = []
    subject_labels = []
    subject_metadata = []
    subject_valid_flags = []
    existing_subject_ids = set()
    
    # Optional resume mode (disabled by default)
    resume_pickle = os.path.join(DATASET_OUTPUT, "als_dataset_complete.pkl")
    if RESUME_EXISTING and os.path.exists(resume_pickle):
        try:
            with open(resume_pickle, 'rb') as f:
                existing = pickle.load(f)
            subject_data = list(existing.get("subject_features", []))
            subject_labels = list(existing.get("subject_labels", []))
            subject_metadata = list(existing.get("subject_metadata", []))
            if "subject_valid_flags" in existing:
                subject_valid_flags = list(existing["subject_valid_flags"])
            else:
                subject_valid_flags = [np.ones((1, sf.shape[1]), dtype=np.int8) for sf in subject_data]
            existing_subject_ids = set(
                m.get("subject_id") for m in subject_metadata if isinstance(m, dict) and m.get("subject_id") is not None
            )
            print(f"\n✅ Resume mode: loaded {len(existing_subject_ids)} subjects. Will skip these subjects.")
        except Exception as e:
            print(f"\n⚠️  Resume mode failed ({e}). Starting fresh.")
            subject_data = []
            subject_labels = []
            subject_metadata = []
            subject_valid_flags = []
            existing_subject_ids = set()
    
    # Group mappings
    # IMPORTANT: keep label mapping consistent with your main experiments
    # Control -> 0, ALSwDysarthria -> 1, ALSwoDysarthria -> 2
    group_label_map = {
        "Control": 0,            # Control
        "ALSwDysarthria": 1,     # Symptomatic ALS
        "ALSwoDysarthria": 2     # Presymptomatic ALS
    }
    
    group_names = {0: "Control", 1: "Symptomatic_ALS", 2: "Presymptomatic_ALS"}
    
    # Track statistics
    stats = {
        "total_subjects": 0,
        "total_utterances": 0,
        "successful_subjects": 0,
        "failed_subjects": [],
        "group_counts": {0: 0, 1: 0, 2: 0}
    }
    
    # Note: when RESUME_EXISTING=False, this run will re-extract all utterances for all subjects.
    
    # Process each group
    for group_folder in os.listdir(BALANCED_DATA_PATH):
        group_path = os.path.join(BALANCED_DATA_PATH, group_folder)
        
        if not os.path.isdir(group_path) or group_folder not in group_label_map:
            continue
        
        label = group_label_map[group_folder]
        print(f"\nProcessing {group_folder} (Label: {label} - {group_names[label]})")
        
        # Get all subjects in this group
        subject_folders = sorted([f for f in os.listdir(group_path) 
                                 if os.path.isdir(os.path.join(group_path, f))])
        
        print(f"Found {len(subject_folders)} subjects")
        
        # Process each subject
        for subject_folder in tqdm(subject_folders, desc=f"{group_folder}"):
            subject_path = os.path.join(group_path, subject_folder)
            subject_id = f"{group_folder}/{subject_folder}"
            if subject_id in existing_subject_ids:
                continue
            
            # Extract subject ID and sex from folder name
            # Format: F1_denoised, M1_denoised, etc.
            base_name = subject_folder.replace('_denoised', '')
            sex = "Female" if base_name.startswith('F') else "Male"
            subj_id_num = base_name[1:] if base_name[0] in ['F', 'M'] else base_name
            
            # Get all WAV files
            wav_files = []
            for file in os.listdir(subject_path):
                if file.lower().endswith('.wav'):
                    wav_files.append(os.path.join(subject_path, file))
            
            if not wav_files:
                print(f"  ⚠️  No WAV files found for {subject_id}")
                stats["failed_subjects"].append(subject_id)
                continue
            
            # Sort files to maintain order
            wav_files = sorted(wav_files)
            
            # Note: subjects may have variable numbers of utterances
            if len(wav_files) != 100:
                print(f"  ⚠️  {subject_id}: Found {len(wav_files)} utterances (not 100). Proceeding anyway.")
            
            # Extract features for all utterances (keep all utterances; fill failures with zeros)
            all_features = []
            valid_flags = []
            failed_files_sample = []
            
            for wav_file in wav_files:  # Process ALL available utterances
                features = extract_opensmile_features(wav_file)
                if features is not None and len(features) == FEATURE_DIM:
                    all_features.append(features)
                    valid_flags.append(1)
                else:
                    all_features.append(np.zeros(FEATURE_DIM, dtype=np.float32))
                    valid_flags.append(0)
                    if len(failed_files_sample) < 5:
                        failed_files_sample.append(os.path.basename(wav_file))
            
            failed_utterances = int(len(wav_files) - sum(valid_flags))
            
            # Convert to numpy array and reshape to (1, num_utterances, num_features)
            features_array = np.array(all_features, dtype=np.float32)
            
            # Reshape to (1, num_utterances, num_features) for subject-wise format
            features_reshaped = np.expand_dims(features_array, axis=0)
            valid_flags_reshaped = np.expand_dims(np.array(valid_flags, dtype=np.int8), axis=0)
            
            # Add to dataset
            subject_data.append(features_reshaped)
            subject_labels.append(label)
            subject_valid_flags.append(valid_flags_reshaped)
            
            # Add metadata
            metadata = {
                "subject_id": subject_id,
                "subject_folder": subject_folder,
                "group": group_folder,
                "group_label": label,
                "group_name": group_names[label],
                "sex": sex,
                "num_utterances": len(all_features),
                "num_utterances_valid": int(sum(valid_flags)),
                "num_utterances_failed": failed_utterances,
                "num_features": features_array.shape[1],
                "original_files": len(wav_files),
                "failed_utterances": failed_utterances,
                "failed_files_sample": failed_files_sample
            }
            subject_metadata.append(metadata)
            
            # Update statistics
            stats["total_subjects"] += 1
            stats["total_utterances"] += len(all_features)
            stats["successful_subjects"] += 1
            stats["group_counts"][label] += 1
            
            if stats["successful_subjects"] % 5 == 0:
                print(f"  Processed {stats['successful_subjects']} subjects...")
    
    # Combine all subject data
    print("\n" + "="*80)
    print("COMBINING DATASET")
    print("="*80)
    
    if not subject_data:
        print("❌ No subjects were successfully processed!")
        return None
    
    # Create dataset dictionary
    dataset = {
        "subject_features": subject_data,  # List of (1, num_utterances, FEATURE_DIM) arrays
        "subject_valid_flags": subject_valid_flags,  # List of (1, num_utterances) validity flags
        "subject_labels": np.array(subject_labels, dtype=np.int32),
        "subject_metadata": subject_metadata,
        "feature_dim": FEATURE_DIM,  # OpenSMILE ComParE13 feature dimension
        "group_names": group_names,
        "group_label_map": group_label_map
    }
    
    # Print summary
    print("\n" + "="*80)
    print("DATASET SUMMARY")
    print("="*80)
    print(f"Total subjects processed: {stats['successful_subjects']}")
    print(f"Total utterances: {stats['total_utterances']}")
    print(f"Average utterances per subject: {stats['total_utterances']/stats['successful_subjects']:.1f}")
    print(f"\nGroup distribution:")
    for label in [0, 1, 2]:
        count = stats["group_counts"][label]
        percentage = (count / stats["successful_subjects"]) * 100
        print(f"  {group_names[label]}: {count} subjects ({percentage:.1f}%)")
    
    print(f"\nSex distribution:")
    sex_counts = {}
    for meta in subject_metadata:
        sex = meta["sex"]
        sex_counts[sex] = sex_counts.get(sex, 0) + 1
    for sex, count in sex_counts.items():
        percentage = (count / stats["successful_subjects"]) * 100
        print(f"  {sex}: {count} subjects ({percentage:.1f}%)")
    
    if stats["failed_subjects"]:
        print(f"\nFailed subjects ({len(stats['failed_subjects'])}):")
        for subj in stats["failed_subjects"][:10]:  # Show first 10
            print(f"  - {subj}")
    
    return dataset, stats

def save_dataset(dataset, stats):
    """
    Save the complete dataset
    """
    print("\n" + "="*80)
    print("SAVING DATASET")
    print("="*80)
    
    if dataset is None or len(dataset.get("subject_features", [])) == 0:
        raise ValueError("Dataset is empty. Run `dataset, stats = extract_all_subjects()` before saving.")
    
    n_subjects = len(dataset["subject_features"])
    total_utts = int(sum(sf.shape[1] for sf in dataset["subject_features"]))
    print(f"Subjects to save: {n_subjects} | Total utterances: {total_utts}")
    
    # Save as compressed numpy file
    dataset_file = os.path.join(DATASET_OUTPUT, "als_opensmile_dataset.npz")
    
    # We need to handle variable-length sequences.
    # IMPORTANT: don't use np.array(list_of_arrays, dtype=object) here because each subject array
    # has a leading dimension of 1 (shape (1, n_utts, FEATURE_DIM)), which can cause NumPy to
    # incorrectly try to form a 2D object array and broadcast.
    subj_feats = dataset["subject_features"]
    subject_features_obj = np.empty(len(subj_feats), dtype=object)
    subject_features_obj[:] = subj_feats
    
    subj_valid = dataset.get("subject_valid_flags", [])
    subject_valid_flags_obj = np.empty(len(subj_valid), dtype=object)
    if len(subj_valid) > 0:
        subject_valid_flags_obj[:] = subj_valid
    np.savez_compressed(
        dataset_file,
        subject_features=subject_features_obj,
        subject_valid_flags=subject_valid_flags_obj,
        subject_labels=dataset["subject_labels"],
        feature_dim=dataset["feature_dim"],
        group_names=list(dataset["group_names"].values()),
        group_label_map=dataset["group_label_map"]
    )
    print(f"✅ Dataset saved to: {dataset_file}")
    
    # Save metadata as CSV
    metadata_file = os.path.join(DATASET_OUTPUT, "subject_metadata.csv")
    metadata_df = pd.DataFrame(dataset["subject_metadata"])
    metadata_df.to_csv(metadata_file, index=False)
    print(f"✅ Metadata saved to: {metadata_file}")
    
    # Save statistics
    stats_file = os.path.join(DATASET_OUTPUT, "extraction_statistics.txt")
    with open(stats_file, 'w') as f:
        f.write("OPENSHMILE FEATURE EXTRACTION STATISTICS\n")
        f.write("="*60 + "\n\n")
        f.write(f"Total subjects attempted: {stats['total_subjects'] + len(stats['failed_subjects'])}\n")
        f.write(f"Successful subjects: {stats['successful_subjects']}\n")
        f.write(f"Failed subjects: {len(stats['failed_subjects'])}\n")
        f.write(f"Total utterances: {stats['total_utterances']}\n")
        f.write(f"Average utterances per subject: {stats['total_utterances']/stats['successful_subjects']:.1f}\n\n")
        
        f.write("GROUP DISTRIBUTION:\n")
        for label in [0, 1, 2]:
            name = dataset["group_names"][label]
            count = stats["group_counts"][label]
            percentage = (count / stats["successful_subjects"]) * 100
            f.write(f"  {name}: {count} subjects ({percentage:.1f}%)\n")
        
        f.write("\nSEX DISTRIBUTION:\n")
        sex_counts = {}
        for meta in dataset["subject_metadata"]:
            sex = meta["sex"]
            sex_counts[sex] = sex_counts.get(sex, 0) + 1
        for sex, count in sex_counts.items():
            percentage = (count / stats["successful_subjects"]) * 100
            f.write(f"  {sex}: {count} subjects ({percentage:.1f}%)\n")
        
        if stats["failed_subjects"]:
            f.write(f"\nFAILED SUBJECTS ({len(stats['failed_subjects'])}):\n")
            for subj in stats["failed_subjects"]:
                f.write(f"  - {subj}\n")
    
    print(f"✅ Statistics saved to: {stats_file}")
    
    # Save as pickle for easy loading
    pickle_file = os.path.join(DATASET_OUTPUT, "als_dataset_complete.pkl")
    with open(pickle_file, 'wb') as f:
        pickle.dump(dataset, f)
    print(f"✅ Complete dataset (pickle) saved to: {pickle_file}")
    
    return dataset_file, metadata_file, stats_file

def create_utterance_level_dataset(dataset):
    """
    Create utterance-level dataset for baseline models
    Each utterance gets the subject's label
    """
    print("\n" + "="*80)
    print("CREATING UTTERANCE-LEVEL DATASET")
    print("="*80)
    
    utterance_features = []
    utterance_labels = []
    utterance_metadata = []
    
    valid_flags_list = dataset.get("subject_valid_flags", None)
    
    for i, (subject_feat, subject_label) in enumerate(zip(dataset["subject_features"], 
                                                         dataset["subject_labels"])):
        # subject_feat shape: (1, num_utterances, features)
        # Remove the first dimension
        subject_utterances = subject_feat[0]  # shape: (num_utterances, features)
        
        for j, utterance_feat in enumerate(subject_utterances):
            utterance_features.append(utterance_feat)
            utterance_labels.append(subject_label)
            
            # Get subject metadata
            meta = dataset["subject_metadata"][i]
            is_valid = int(valid_flags_list[i][0][j]) if valid_flags_list is not None else 1
            utterance_meta = {
                "subject_id": meta["subject_id"],
                "utterance_index": j,
                "group": meta["group"],
                "group_label": subject_label,
                "sex": meta["sex"],
                "is_valid": is_valid,
                "total_utterances": meta["num_utterances"]
            }
            utterance_metadata.append(utterance_meta)
    
    # Convert to numpy arrays
    utterance_features = np.array(utterance_features, dtype=np.float32)
    utterance_labels = np.array(utterance_labels, dtype=np.int32)
    
    print(f"Utterance dataset created:")
    print(f"  - Total utterances: {len(utterance_features)}")
    print(f"  - Features shape: {utterance_features.shape}")
    print(f"  - Labels shape: {utterance_labels.shape}")
    
    # Save utterance dataset
    utterance_file = os.path.join(DATASET_OUTPUT, "utterance_level_dataset.npz")
    np.savez_compressed(
        utterance_file,
        features=utterance_features,
        labels=utterance_labels,
        feature_dim=dataset["feature_dim"],
        group_names=list(dataset["group_names"].values())
    )
    print(f"✅ Utterance-level dataset saved to: {utterance_file}")
    
    # Save utterance metadata
    utterance_meta_file = os.path.join(DATASET_OUTPUT, "utterance_metadata.csv")
    pd.DataFrame(utterance_metadata).to_csv(utterance_meta_file, index=False)
    print(f"✅ Utterance metadata saved to: {utterance_meta_file}")
    
    return utterance_features, utterance_labels

def load_and_verify_dataset():
    """
    Load and verify the saved dataset
    """
    print("\n" + "="*80)
    print("VERIFYING DATASET")
    print("="*80)
    
    # Load pickle file
    pickle_file = os.path.join(DATASET_OUTPUT, "als_dataset_complete.pkl")
    
    if not os.path.exists(pickle_file):
        print("❌ Dataset not found!")
        return None
    
    with open(pickle_file, 'rb') as f:
        dataset = pickle.load(f)
    
    print(f"Dataset loaded successfully!")
    print(f"Number of subjects: {len(dataset['subject_features'])}")
    print(f"Number of labels: {len(dataset['subject_labels'])}")
    print(f"Feature dimension: {dataset['feature_dim']}")
    
    # Check a few subjects
    print(f"\nSample subject information:")
    for i in range(min(3, len(dataset['subject_features']))):
        feat_shape = dataset['subject_features'][i].shape
        label = dataset['subject_labels'][i]
        metadata = dataset['subject_metadata'][i]
        
        print(f"Subject {i}:")
        print(f"  Features shape: {feat_shape}")
        print(f"  Label: {label} ({dataset['group_names'][label]})")
        print(f"  Subject ID: {metadata['subject_id']}")
        print(f"  Sex: {metadata['sex']}")
        print(f"  Utterances: {metadata['num_utterances']} (valid={metadata.get('num_utterances_valid','?')}, failed={metadata.get('num_utterances_failed','?')}, files={metadata.get('original_files','?')})")
        print()
    
    return dataset

# ============================================
# MAIN EXECUTION
# ============================================

print("\n" + "="*80)
print("STEP 2: COMPLETE DATASET CREATION")
print("="*80)
print("\nThis will:")
print("1. Process ALL subjects")
print("2. Extract OpenSMILE features for each utterance")
print("3. Create subject-wise dataset (1, num_utterances, FEATURE_DIM)")
print("4. Create utterance-level dataset")
print("5. Save all files for classification")
print("\nNote: This extracts ALL available utterances per subject (variable-length).")
print("If you want to resume a partially-finished run, set RESUME_EXISTING=True (otherwise it will overwrite outputs).")
print("="*80)

response = input("\nProceed with full extraction? This may take several hours. (yes/no): ").lower()

if response not in ['yes', 'y']:
    print("Operation cancelled.")
    exit(0)

# Execute STEP 2
try:
    # Extract all subjects
    dataset, stats = extract_all_subjects()
    
    if dataset is None:
        print("❌ Dataset creation failed!")
        exit(1)
    
    # Save dataset
    dataset_file, metadata_file, stats_file = save_dataset(dataset, stats)
    
    # Create utterance-level dataset
    utterance_features, utterance_labels = create_utterance_level_dataset(dataset)
    
    # Verify dataset
    loaded_dataset = load_and_verify_dataset()
    
    print("\n" + "="*80)
    print("✅ STEP 2 COMPLETED SUCCESSFULLY!")
    print("="*80)
    print("\nFiles created:")
    print(f"1. Subject-wise dataset: {DATASET_OUTPUT}/als_opensmile_dataset.npz")
    print(f"2. Subject metadata: {DATASET_OUTPUT}/subject_metadata.csv")
    print(f"3. Utterance-level dataset: {DATASET_OUTPUT}/utterance_level_dataset.npz")
    print(f"4. Utterance metadata: {DATASET_OUTPUT}/utterance_metadata.csv")
    print(f"5. Statistics: {DATASET_OUTPUT}/extraction_statistics.txt")
    print(f"6. Complete dataset: {DATASET_OUTPUT}/als_dataset_complete.pkl")
    
    print("\n" + "="*80)
    print("READY FOR CLASSIFICATION!")
    print("="*80)
except KeyboardInterrupt:
    print("\n\n⚠️  Extraction interrupted by user!")
except Exception as e:
    print(f"\n❌ Error during extraction: {e}")
    import traceback
    traceback.print_exc()
